In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score,f1_score,recall_score,classification_report, confusion_matrix
from sklearn.feature_selection import chi2,SelectKBest,mutual_info_classif,RFE

In [4]:
df=pd.read_csv(r"C:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\data\medical_disease_data_cleaning.csv")
df

,Age,Gender,BMI,BloodPressure,GlucoseLevel,Cholesterol,HeartRate,Smoking,Alcohol,PhysicalActivity,FamilyHistory,Disease
0,23,1,17.5,102,177,161,88,0,0,2,0,1
1,69,1,31.1,118,256,147,92,0,0,0,1,4
2,61,0,31.3,114,206,193,74,1,0,2,0,1
3,47,0,23.9,105,162,253,68,0,0,1,0,1
4,47,1,28.3,148,121,280,84,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,49,1,27.1,177,183,268,59,0,0,0,0,3
9996,84,1,29.2,142,215,145,77,0,1,0,1,3
9997,41,1,22.8,177,265,133,71,0,0,2,1,3
9998,55,1,29.0,126,241,298,57,0,0,2,1,4


## divide data in input featuers and target 

In [12]:
x=df.drop('Disease',axis=1)
y=df["Disease"]

In [10]:
x

,Age,Gender,BMI,BloodPressure,GlucoseLevel,Cholesterol,HeartRate,Smoking,Alcohol,PhysicalActivity,FamilyHistory
0,23,1,17.5,102,177,161,88,0,0,2,0
1,69,1,31.1,118,256,147,92,0,0,0,1
2,61,0,31.3,114,206,193,74,1,0,2,0
3,47,0,23.9,105,162,253,68,0,0,1,0
4,47,1,28.3,148,121,280,84,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,49,1,27.1,177,183,268,59,0,0,0,0
9996,84,1,29.2,142,215,145,77,0,1,0,1
9997,41,1,22.8,177,265,133,71,0,0,2,1
9998,55,1,29.0,126,241,298,57,0,0,2,1


In [11]:
y

,Disease
0,1
1,4
2,1
3,1
4,1
...,...
9995,3
9996,3
9997,3
9998,4


## spliting data in training and testing

In [14]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train.shape,x_test.shape,y_train.shape,y_test.shape

((8000, 11), (2000, 11), (8000,), (2000,))

## Select best features using feature selection techniques

#### 1. Chi2

In [16]:
chi_score, p_value=chi2(x_train,y_train)
chi_result = pd.DataFrame({
    'Feature': x_train.columns,
    'Chi2_Score': chi_score,
    'P_Value': p_value
})


print(chi_result.sort_values('Chi2_Score',ascending=False))


             Feature   Chi2_Score       P_Value
4       GlucoseLevel  9637.292780  0.000000e+00
5        Cholesterol  7352.598335  0.000000e+00
0                Age  2748.205505  0.000000e+00
3      BloodPressure  2335.595129  0.000000e+00
7            Smoking   384.171284  7.311655e-82
10     FamilyHistory   350.236034  1.559608e-74
2                BMI   263.208461  9.280391e-56
6          HeartRate     7.041426  1.337152e-01
8            Alcohol     6.180366  1.860772e-01
9   PhysicalActivity     0.477781  9.756277e-01
1             Gender     0.398869  9.825694e-01


### 2. mutual information 

In [17]:
mi_score=mutual_info_classif(x_train,y_train)
mi_result=pd.DataFrame({
    'Feature':x_train.columns,
    'MI_Score':mi_score
})


print(mi_result.sort_values('MI_Score',ascending=False))


             Feature  MI_Score
2                BMI  0.048798
4       GlucoseLevel  0.044043
5        Cholesterol  0.039575
7            Smoking  0.039024
0                Age  0.037017
10     FamilyHistory  0.029766
3      BloodPressure  0.026365
9   PhysicalActivity  0.008037
8            Alcohol  0.000890
1             Gender  0.000000
6          HeartRate  0.000000


### Select best 8 features using Chi2 and SelectKBest(Filter Method)

In [18]:
selector= SelectKBest(score_func=chi2,k=8)
x_train_chi =selector.fit_transform(x_train,y_train)
x_test_chi=selector.transform(x_test)


selected_features = x_train.columns[selector.get_support()]

selected_col=pd.DataFrame({
    'Features':selected_features
})
print("Selected Features:\n")

print(selected_col)


Selected Features:

        Features
0            Age
1            BMI
2  BloodPressure
3   GlucoseLevel
4    Cholesterol
5      HeartRate
6        Smoking
7  FamilyHistory


### Select best 8 features using RFE (Recursive Feature Elimination - Wrapper Method)

In [24]:
model =RandomForestClassifier()
rfe = RFE(estimator=model,n_features_to_select=8)
rfe.fit(x_train,y_train)

,"estimator estimator: ``Estimator`` instanceA supervised learning estimator with a ``fit`` method that providesinformation about feature importance(e.g. `coef_`, `feature_importances_`).",RandomForestClassifier()
,"n_features_to_select n_features_to_select: int or float, default=NoneThe number of features to select. If `None`, half of the features areselected. If integer, the parameter is the absolute number of featuresto select. If float between 0 and 1, it is the fraction of features toselect... versionchanged:: 0.24 Added float values for fractions.",8
,"step step: int or float, default=1If greater than or equal to 1, then ``step`` corresponds to the(integer) number of features to remove at each iteration.If within (0.0, 1.0), then ``step`` corresponds to the percentage(rounded down) of features to remove at each iteration.",1
,"verbose verbose: int, default=0Controls verbosity of output.",0
,"importance_getter importance_getter: str or callable, default='auto'If 'auto', uses the feature importance either through a `coef_`or `feature_importances_` attributes of estimator.Also accepts a string that specifies an attribute name/pathfor extracting feature importance (implemented with `attrgetter`).For example, give `regressor_.coef_` in case of:class:`~sklearn.compose.TransformedTargetRegressor` or`named_steps.clf.feature_importances_` in case ofclass:`~sklearn.pipeline.Pipeline` with its last step named `clf`.If `callable`, overrides the default feature importance getter.The callable is passed with the fitted estimator and it shouldreturn importance for each feature... versionadded:: 0.24",'auto'
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only available when `estimator` is a classifier.","ndarray[int64](5,)","[0,1,2,3,4]"
estimator_ estimator_: ``Estimator`` instanceThe fitted estimator used to select features.,RandomForestClassifier,RandomForestClassifier()
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](11,)","['Age','Gender','BMI',...,'Alcohol','PhysicalActivity','FamilyHistory']"
n_features_ n_features_: intThe number of selected features.,int64,np.int64(8)
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,11


In [ ]:
## features  ranking using RFE

rfe_result = pd.DataFrame({
    "features_name": x_train.columns,
    "Features_Ranking": rfe.ranking_
})

print(rfe_result.sort_values("Features_Ranking",ascending=True))

       features_name  Features_Ranking
0                Age                 1
2                BMI                 1
3      BloodPressure                 1
4       GlucoseLevel                 1
6          HeartRate                 1
5        Cholesterol                 1
7            Smoking                 1
10     FamilyHistory                 1
9   PhysicalActivity                 2
1             Gender                 3
8            Alcohol                 4


In [ ]:
### Selected Features using RFE

selected_features_rfe = pd.DataFrame({
    "Features": x_train.columns[rfe.support_]
    })

print(selected_features_rfe)

        Features
0            Age
1            BMI
2  BloodPressure
3   GlucoseLevel
4    Cholesterol
5      HeartRate
6        Smoking
7  FamilyHistory


In [27]:
x_train_rfe = rfe.transform(x_train)
x_test_rfe = rfe.transform(x_test)

print(x_train_rfe.shape)
print(x_test_rfe.shape)

(8000, 8)
(2000, 8)


## Now Train the model

#### 👉  Train and Evaluate Model using Chi2 Selected Features

In [ ]:
models={
    "👉 logistic_regression":LogisticRegression(),
    "👉 decision_tree":DecisionTreeClassifier(),   
    "👉 random_forest":RandomForestClassifier(),
    "👉 svm":SVC(),
    "👉 knn":KNeighborsClassifier()
}

In [42]:
for name, model in models.items():
    model.fit(x_train_chi, y_train)
    y_pred_chi = model.predict(x_test_chi)

    print("\n",name)

    print("Accuracy Score_chi:", accuracy_score(y_test, y_pred_chi))
    print("precision score_chi:", precision_score(y_test, y_pred_chi,average='weighted'))
    print("Recall Score_chi:", recall_score(y_test, y_pred_chi,average='weighted'))
    print("F1 Score_chi:", f1_score(y_test, y_pred_chi,average='weighted'))
    print("Confusion Matrix_chi:\n", confusion_matrix(y_test, y_pred_chi))
    print("Classification Report_chi:\n", classification_report(y_test, y_pred_chi))

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"


 👉 logistic_regression
Accuracy Score_chi: 0.6415
precision score_chi: 0.4753435572865953
Recall Score_chi: 0.6415
F1 Score_chi: 0.528340280725779
Confusion Matrix_chi:
 [[   0   34    0   12    0]
 [   0 1252    0   27    0]
 [   0   42    0   11    0]
 [   0  325    0   31    0]
 [   0  243    0   23    0]]
Classification Report_chi:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        46
           1       0.66      0.98      0.79      1279
           2       0.00      0.00      0.00        53
           3       0.30      0.09      0.13       356
           4       0.00      0.00      0.00       266

    accuracy                           0.64      2000
   macro avg       0.19      0.21      0.18      2000
weighted avg       0.48      0.64      0.53      2000


 👉 decision_tree
Accuracy Score_chi: 0.655
precision score_chi: 0.6534217269600809
Recall Score_chi: 0.655
F1 Score_chi: 0.6539239754797874
Confusion Matrix_chi:
 [[  14 

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

### ## Train and Evaluate Model using RFE Selected Features

In [43]:
for name, model in models.items():
    model.fit(x_train_rfe, y_train)
    y_pred_rfe = model.predict(x_test_rfe)

    print("\n",name)

    print("Accuracy Score_rfe:", accuracy_score(y_test, y_pred_rfe))
    print("precision score_rfe:", precision_score(y_test, y_pred_rfe,average='weighted'))
    print("Recall Score_rfe:", recall_score(y_test, y_pred_rfe,average='weighted'))
    print("F1 Score_rfe:", f1_score(y_test, y_pred_rfe,average='weighted'))
    print("Confusion Matrix_rfe:\n", confusion_matrix(y_test, y_pred_rfe))
    print("Classification Report_rfe:\n", classification_report(y_test, y_pred_rfe))

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"


 👉 logistic_regression
Accuracy Score_rfe: 0.6415
precision score_rfe: 0.4753435572865953
Recall Score_rfe: 0.6415
F1 Score_rfe: 0.528340280725779
Confusion Matrix_rfe:
 [[   0   34    0   12    0]
 [   0 1252    0   27    0]
 [   0   42    0   11    0]
 [   0  325    0   31    0]
 [   0  243    0   23    0]]
Classification Report_rfe:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        46
           1       0.66      0.98      0.79      1279
           2       0.00      0.00      0.00        53
           3       0.30      0.09      0.13       356
           4       0.00      0.00      0.00       266

    accuracy                           0.64      2000
   macro avg       0.19      0.21      0.18      2000
weighted avg       0.48      0.64      0.53      2000


 👉 decision_tree
Accuracy Score_rfe: 0.6515
precision score_rfe: 0.6505190003335853
Recall Score_rfe: 0.6515
F1 Score_rfe: 0.6507853286742826
Confusion Matrix_rfe:
 [[  1

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

## Handling Class Imbalance

In [44]:
models={
    "👉 logistic_regression":LogisticRegression(),
    "👉 decision_tree":DecisionTreeClassifier(random_state=42,class_weight='balanced'),   
    "👉 random_forest":RandomForestClassifier(random_state=42,class_weight='balanced'),
    "👉 svm":SVC(),
    "👉 knn":KNeighborsClassifier()
}

### Model Training using Chi-Square Selected Features


In [45]:
for name, model in models.items():
    model.fit(x_train_chi, y_train)
    y_pred_chi = model.predict(x_test_chi)

    print("\n",name)

    print("Accuracy Score_chi:", accuracy_score(y_test, y_pred_chi))
    print("precision score_chi:", precision_score(y_test, y_pred_chi,average='weighted'))
    print("Recall Score_chi:", recall_score(y_test, y_pred_chi,average='weighted'))
    print("F1 Score_chi:", f1_score(y_test, y_pred_chi,average='weighted'))
    print("Confusion Matrix_chi:\n", confusion_matrix(y_test, y_pred_chi))
    print("Classification Report_chi:\n", classification_report(y_test, y_pred_chi))

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"


 👉 logistic_regression
Accuracy Score_chi: 0.6415
precision score_chi: 0.4753435572865953
Recall Score_chi: 0.6415
F1 Score_chi: 0.528340280725779
Confusion Matrix_chi:
 [[   0   34    0   12    0]
 [   0 1252    0   27    0]
 [   0   42    0   11    0]
 [   0  325    0   31    0]
 [   0  243    0   23    0]]
Classification Report_chi:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        46
           1       0.66      0.98      0.79      1279
           2       0.00      0.00      0.00        53
           3       0.30      0.09      0.13       356
           4       0.00      0.00      0.00       266

    accuracy                           0.64      2000
   macro avg       0.19      0.21      0.18      2000
weighted avg       0.48      0.64      0.53      2000


 👉 decision_tree
Accuracy Score_chi: 0.658
precision score_chi: 0.6547219431950854
Recall Score_chi: 0.658
F1 Score_chi: 0.6563051564259224
Confusion Matrix_chi:
 [[  14 

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

### Model Training using RFE Selected Features

In [46]:
for name, model in models.items():
    model.fit(x_train_rfe, y_train)
    y_pred_rfe = model.predict(x_test_rfe)

    print("\n",name)

    print("Accuracy Score_rfe:", accuracy_score(y_test, y_pred_rfe))
    print("precision score_rfe:", precision_score(y_test, y_pred_rfe,average='weighted'))
    print("Recall Score_rfe:", recall_score(y_test, y_pred_rfe,average='weighted'))
    print("F1 Score_rfe:", f1_score(y_test, y_pred_rfe,average='weighted'))
    print("Confusion Matrix_rfe:\n", confusion_matrix(y_test, y_pred_rfe))
    print("Classification Report_rfe:\n", classification_report(y_test, y_pred_rfe))

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"


 👉 logistic_regression
Accuracy Score_rfe: 0.6415
precision score_rfe: 0.4753435572865953
Recall Score_rfe: 0.6415
F1 Score_rfe: 0.528340280725779
Confusion Matrix_rfe:
 [[   0   34    0   12    0]
 [   0 1252    0   27    0]
 [   0   42    0   11    0]
 [   0  325    0   31    0]
 [   0  243    0   23    0]]
Classification Report_rfe:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        46
           1       0.66      0.98      0.79      1279
           2       0.00      0.00      0.00        53
           3       0.30      0.09      0.13       356
           4       0.00      0.00      0.00       266

    accuracy                           0.64      2000
   macro avg       0.19      0.21      0.18      2000
weighted avg       0.48      0.64      0.53      2000


 👉 decision_tree
Accuracy Score_rfe: 0.658
precision score_rfe: 0.6547219431950854
Recall Score_rfe: 0.658
F1 Score_rfe: 0.6563051564259224
Confusion Matrix_rfe:
 [[  14 

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

### Model Training using All Features

In [47]:
for name, model in models.items():
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)

    print("\n",name)

    print("Accuracy Score:", accuracy_score(y_test, y_pred))
    print("precision score:", precision_score(y_test, y_pred,average='weighted'))
    print("Recall Score:", recall_score(y_test, y_pred,average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred,average='weighted'))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"


 👉 logistic_regression
Accuracy Score: 0.6415
precision score: 0.4668518897414038
Recall Score: 0.6415
F1 Score: 0.5138076901129983
Confusion Matrix:
 [[   0   37    0    9    0]
 [   0 1270    0    9    0]
 [   0   47    0    6    0]
 [   0  343    0   13    0]
 [   0  258    0    8    0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00        46
           1       0.65      0.99      0.79      1279
           2       0.00      0.00      0.00        53
           3       0.29      0.04      0.06       356
           4       0.00      0.00      0.00       266

    accuracy                           0.64      2000
   macro avg       0.19      0.21      0.17      2000
weighted avg       0.47      0.64      0.51      2000


 👉 decision_tree
Accuracy Score: 0.6365
precision score: 0.6347332912207181
Recall Score: 0.6365
F1 Score: 0.6355054443561928
Confusion Matrix:
 [[  14    6   12    8    6]
 [   1 1067    7  10

c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\RIya\Desktop\6month-Data-Science\machine learning\medical\medical_project\medical\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 

## Final model training

In [56]:
final_model=random_forest=RandomForestClassifier(random_state=42,class_weight='balanced')
final_model.fit(x_train_chi, y_train)


,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.

In [ ]:
final_y_pred = final_model.predict(x_test_chi)

In [ ]:


print("Accuracy Score_final:", accuracy_score(y_test, final_y_pred))
print("precision score_final:", precision_score(y_test, final_y_pred,average='weighted'))
print("Recall Score_final:", recall_score(y_test, final_y_pred,average='weighted'))
print("F1 Score_final:", f1_score(y_test, final_y_pred,average='weighted'))
print("Confusion Matrix_final:\n", confusion_matrix(y_test, final_y_pred))
print("Classification Report_final:\n", classification_report(y_test, final_y_pred))

Accuracy Score_final: 0.665
precision score_final: 0.7365198016219666
Recall Score_final: 0.665
F1 Score_final: 0.6911274555093251
Confusion Matrix_final:
 [[  19    0    9   17    1]
 [   0 1006    0  131  142]
 [  12    0    8   32    1]
 [  15   18   12  178  133]
 [   0   24    0  123  119]]
Classification Report_final:
               precision    recall  f1-score   support

           0       0.41      0.41      0.41        46
           1       0.96      0.79      0.86      1279
           2       0.28      0.15      0.20        53
           3       0.37      0.50      0.43       356
           4       0.30      0.45      0.36       266

    accuracy                           0.67      2000
   macro avg       0.46      0.46      0.45      2000
weighted avg       0.74      0.67      0.69      2000



In [57]:
import joblib
moodel=joblib.dump(final_model, 'medical_disease_prediction_model.lb')